# Imports

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import re
from collections import Counter
import pandas as pd
from sklearn import preprocessing

# Tokenization + vocab

In [2]:
def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text.split()


def build_vocab(texts, max_vocab=10000):
    counter = Counter()
    for t in texts:
        counter.update(tokenize(t))

    vocab = {"<PAD>": 0, "<UNK>": 1}
    for i, (w, _) in enumerate(counter.most_common(max_vocab - 2), start=2):
        vocab[w] = i
    return vocab


# Encode text (indices only)

In [3]:
def encode(text, vocab, max_len=50):
    tokens = tokenize(text)
    ids = [vocab.get(tok, vocab["<UNK>"]) for tok in tokens]
    ids = ids[:max_len]
    ids += [vocab["<PAD>"]] * (max_len - len(ids))
    return ids


# Dataset

In [4]:
train_data = pd.read_csv("/content/Corona_NLP_train.csv", encoding='latin-1')
test_data = pd.read_csv("/content/Corona_NLP_test.csv", encoding='latin-1')
train_data = train_data.drop(["UserName", "ScreenName", "Location", "TweetAt"], axis=1)
test_data = test_data.drop(["UserName", "ScreenName", "Location", "TweetAt"], axis=1)


In [5]:
le = preprocessing.LabelEncoder()
le.fit(train_data['Sentiment'])
train_data['Sentiment'] = le.transform(train_data['Sentiment'])
test_data['Sentiment'] = le.transform(test_data['Sentiment'])

In [6]:
X_train = train_data['OriginalTweet']
y_train = train_data['Sentiment']
X_test = test_data['OriginalTweet']
y_test = test_data['Sentiment']

In [7]:
len(X_train)

41157

In [8]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=50):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = torch.tensor(
            encode(self.texts[idx], self.vocab, self.max_len),
            dtype=torch.long
        )
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y


# Manual Embedding Layer

In [9]:
class ManualEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim

        self.weight = nn.Parameter(
            torch.randn(vocab_size, embed_dim) * 0.01
        )

    def forward(self, x):
        # # x: [batch, seq_len]
        # one_hot = torch.nn.functional.one_hot(
        #     x, num_classes=self.vocab_size
        # ).float()
        # # [batch, seq_len, vocab_size]

        # embedded = torch.matmul(one_hot, self.weight)
        # # [batch, seq_len, embed_dim]

        # return embedded
        return self.weight[x]


# Manual LSTM Cell

In [10]:
class ManualLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim

        self.W_i = nn.Linear(input_dim + hidden_dim, hidden_dim)
        self.W_f = nn.Linear(input_dim + hidden_dim, hidden_dim)
        self.W_g = nn.Linear(input_dim + hidden_dim, hidden_dim)
        self.W_o = nn.Linear(input_dim + hidden_dim, hidden_dim)

    def forward(self, x, h_prev, c_prev):
        combined = torch.cat([x, h_prev], dim=1)

        i = torch.sigmoid(self.W_i(combined))
        f = torch.sigmoid(self.W_f(combined))
        g = torch.tanh(self.W_g(combined))
        o = torch.sigmoid(self.W_o(combined))

        c = f * c_prev + i * g
        h = o * torch.tanh(c)

        return h, c


# FULL Model

In [11]:
class LSTMSentiment(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()

        self.embedding = ManualEmbedding(vocab_size, embed_dim)
        self.lstm_cell = ManualLSTMCell(embed_dim, hidden_dim)

        self.fc = nn.Linear(hidden_dim, 5)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        batch_size, seq_len = x.size()
        device = x.device

        h = torch.zeros(batch_size, self.lstm_cell.hidden_dim).to(device)
        c = torch.zeros(batch_size, self.lstm_cell.hidden_dim).to(device)

        x = self.embedding(x)

        for t in range(seq_len):
            h, c = self.lstm_cell(x[:, t, :], h, c)

        out = self.fc(h)
        # return self.sigmoid(out).squeeze()
        return out.squeeze()


# Training & Evaluation

In [12]:
def train(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        preds = model(x)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = torch.argmax(model(x), dim=1)
            # preds = (model(x) > 0.5).float()
            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total


# Running

In [14]:
# texts = [
#     "this movie was amazing",
#     "i hated this movie",
#     "fantastic story and acting",
#     "worst film ever"
# ]
# labels = [1, 0, 1, 0]

# vocab = build_vocab(texts)
vocab = build_vocab(X_train)


# dataset = SentimentDataset(texts, labels, vocab)
dataset = SentimentDataset(X_train, y_train, vocab)
loader = DataLoader(dataset, batch_size=256, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LSTMSentiment(
    vocab_size=len(vocab),
    embed_dim=64,
    hidden_dim=64
).to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(5):
    loss = train(model, loader, optimizer, criterion, device)
    acc = evaluate(model, loader, device)
    print(f"Epoch {epoch+1} | Loss: {loss:.4f} | Acc: {acc:.4f}")


Epoch 1 | Loss: 1.5684 | Acc: 0.3071
Epoch 2 | Loss: 1.3869 | Acc: 0.4820
Epoch 3 | Loss: 1.1884 | Acc: 0.5408
Epoch 4 | Loss: 1.1029 | Acc: 0.5984
Epoch 5 | Loss: 1.0191 | Acc: 0.6405


In [15]:
test_dataset = SentimentDataset(X_test, y_test, vocab)
test_loader = DataLoader(test_dataset, batch_size=256)
acc = evaluate(model, test_loader, device)
acc

0.5229067930489731

In [ ]:
device

device(type='cuda')